In [3]:
# Installation

!pip install langchain==0.3.25 langchain-groq langchain-community \
langchain-text-splitters langchain-huggingface chromadb pypdf \
sentence-transformers gradio -q

In [ ]:
# Version check

import langchain
import chromadb
import gradio
print("LangChain version:", langchain.__version__)
print("ChromaDB version:", chromadb.__version__)
print("Gradio version:", gradio.__version__)
print("All packages installed successfully!")

In [ ]:
# API key

import os
os.environ["GROQ_API_KEY"] = "paste-your-api_key"
print("Groq API Key set!")

In [ ]:
# Imports

from langchain_groq import ChatGroq
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain.chains import RetrievalQA
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr
import os
print("All imports successful!")

In [ ]:
# Upload file

from google.colab import files
uploaded = files.upload()
filename = list(uploaded.keys())[0]

loader = PyPDFLoader(filename)
documents = loader.load()
print(f"Pages loaded: {len(documents)}")

In [ ]:
# Chunk + Embed

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks = text_splitter.split_documents(documents)
print(f"Chunks created: {len(chunks)}")

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./rag_db"
)
print(f"Vectors stored: {vectorstore._collection.count()}")

In [ ]:
# Build RAG chain

llm = ChatGroq(
    model="llama-3.1-8b-instant",  # free and fast!
    temperature=0.3
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True
)
print("RAG chain ready!")

In [ ]:
# Test

query = "What is this document about?"

result = qa_chain.invoke({"query": query})
print("🤖 Answer:", result["result"])
print("\n📄 Sources used:", len(result["source_documents"]), "chunks")